In [16]:
import os
import sys

import torch
import numpy as np
import pandas as pd

import shared.utils as su
from utils.video import read_frames_decord
from notebooks.eval_care_retrieval import (
    load_data,
    compute_metrics,
    print_metrics_as_latex_row,
)
import json

In [18]:
from models.qwen3vl import AutoEncoder

model_path = "/work/piyush/pretrained_checkpoints/Qwen3-VL-4B-Instruct"
# model_path = "/work/piyush/experiments/CaRe/Qwen3-VL-4B-Instruct/final-10112025/nli_9000+ego_1000+subj_replaced-seed_42"
encoder = AutoEncoder.from_pretrained(
    model_path, dtype=torch.bfloat16, device_map="auto", attn_implementation="flash_attention_2",
)
frames = read_frames_decord(video_path='../assets/demo.mp4', num_frames=16)
text = "This video features a man slicing tomatoes in the kitchen."
vision_emb = encoder.encode_vision(frames.unsqueeze(0))
text_emb = encoder.encode_text(text)
print(f'Vision embedding shape: {vision_emb.shape}')
print(f'Text embedding shape: {text_emb.shape}')
# print(f'Cosine similarity: {cosine_similarity(vision_emb, text_emb)}')

Loading EncoderForQwen3VL from /work/piyush/pretrained_checkpoints/Qwen3-VL-4B-Instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Vision embedding shape: torch.Size([1, 2560])
Text embedding shape: torch.Size([1, 2560])


In [19]:
# Load data
dataset = "ssv2"
df = load_data(dataset=dataset)
df = df.drop_duplicates(subset=['id', 'text_id']).reset_index(drop=True)
df.shape

Number of rows:  1430
Sample row: 
{
    "id": 69703,
    "label": "moving pen up",
    "template": "Moving [something] up",
    "placeholders": "['pen']",
    "target": 114,
    "chiral_label": 0.0,
    "chiral_triplet_id": "3f20f09b",
    "noun": "['something']",
    "text_id": "3f20f09b_0.0",
    "video_path": "/scratch/shared/beegfs/piyush/datasets/SSv2/20bn-something-something-v2/69703.webm"
}


(1430, 10)

In [20]:
# Compute text features
text_ids = df['text_id'].unique()
texts_feat = {}
j = 0
for text_id in su.log.tqdm_iterator(text_ids, desc='Computing text features'):
    text = df[df.text_id == text_id].template.unique()[0]
    zt = encoder.encode_text(text).squeeze(0)
    zt = torch.nn.functional.normalize(zt, dim=-1)
    texts_feat[text_id] = zt.cpu().float()
    if j == 0:
        print("Text embedding: ", zt.shape)
    j += 1
len(texts_feat)

Computing text features:   0%|          | 0/32 [00:00<?, ?it/s]

Text embedding:  torch.Size([2560])


32

In [21]:
# Compute video features
video_paths = df.video_path.unique()
video_ids = df.id.unique()
video_feat = {}
j = 0
for video_path in su.log.tqdm_iterator(video_paths, desc='Computing video features'):
    frames = read_frames_decord(video_path=video_path, num_frames=16).unsqueeze(0)
    zv = encoder.encode_vision(frames)[0]
    zv = torch.nn.functional.normalize(zv, dim=-1)
    video_feat[video_ids[j]] = zv.cpu().float()
    if j == 0:
        print("Video embedding: ", zv.shape)
    j += 1


Computing video features:   0%|          | 0/1430 [00:00<?, ?it/s]

Video embedding:  torch.Size([2560])


In [22]:
metrics = compute_metrics(df, video_feat, texts_feat, show_metrics=False)
print_metrics_as_latex_row(metrics, sep='& ')

# Save metrics
save_metrics = True
if save_metrics:
    save_dir = os.path.join(model_path, 'metrics')
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f'metrics-{dataset}.json'), 'w') as f:
        json.dump(metrics, f)
else:
    print(f"Metrics not saved.")
    print(json.dumps(metrics, indent=4))


save_embs = True
if save_embs:
    save_dir = os.path.join(model_path, 'embs')
    os.makedirs(save_dir, exist_ok=True)
    print(f"Saving embeddings to {save_dir}")
    torch.save(video_feat, os.path.join(save_dir, f'video_feat-{dataset}.pt'))
    torch.save(texts_feat, os.path.join(save_dir, f'texts_feat-{dataset}.pt'))

56.1& 23.8& 15.9& 59.9& 16.9& 10.3
Saving embeddings to /work/piyush/pretrained_checkpoints/Qwen3-VL-4B-Instruct/embs
